In [3]:
import numpy as np
from tensorflow.keras import utils as np_utils
from tensorflow.keras.callbacks import ModelCheckpoint

In [4]:
import numpy as np

X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1.npy")
X1=X1.reshape(X1.shape[0], 1, X1.shape[2], X1.shape[1])

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2.npy")
X2=X2.reshape(X2.shape[0], 1, X2.shape[2], X2.shape[1])

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3.npy")
X3=X3.reshape(X3.shape[0], 1, X3.shape[2], X3.shape[1])

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4.npy")
X4=X4.reshape(X4.shape[0], 1, X4.shape[2], X4.shape[1])

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5.npy")
X5=X5.reshape(X5.shape[0], 1, X5.shape[2], X5.shape[1])

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6.npy")
X6=X6.reshape(X6.shape[0], 1, X6.shape[2], X6.shape[1])

X_train=np.append(X1, X2,0)
X_train=np.append(X_train, X3,0)
X_train=np.append(X_train, X4,0)
X_train=np.append(X_train, X5,0)
X_train=np.append(X_train, X6,0)

X_train.shape

X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1L.npy")

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2L.npy")

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3L.npy")

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4L.npy")

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5L.npy")

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6L.npy")

y_train=np.append(X1, X2,0)
y_train=np.append(y_train, X3,0)
y_train=np.append(y_train, X4,0)
y_train=np.append(y_train, X5,0)
y_train=np.append(y_train, X6,0)

y_train.shape

X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7.npy")
X7=X7.reshape(X7.shape[0], 1, X7.shape[2], X7.shape[1])

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8.npy")
X8=X8.reshape(X8.shape[0], 1, X8.shape[2], X8.shape[1])

X_val=np.append(X7, X8,0)

X_val.shape

X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7L.npy")

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8L.npy")

y_val=np.append(X7, X8,0)

y_val.shape

X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9.npy")
X9=X9.reshape(X9.shape[0], 1, X9.shape[2], X9.shape[1])
X_test=X9
X_test.shape

X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9L.npy")
y_test=X9
y_test.shape


(576,)

In [5]:
X_train.shape

(3456, 1, 1001, 22)

In [6]:
chans=22
samples=1001
kernels=1

In [ ]:

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.layers import Conv2D, AveragePooling2D
from tensorflow.keras.layers import SeparableConv2D, DepthwiseConv2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import SpatialDropout2D
from tensorflow.keras.layers import Input, Flatten
from tensorflow.keras.constraints import max_norm

def EEGNet(nb_classes, Chans = 64, Samples = 128, 
             dropoutRate = 0.5, kernLength = 64, F1 = 8, 
             D = 2, F2 = 16, norm_rate = 0.25, dropoutType = 'Dropout'):
    
    
    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError('dropoutType must be one of SpatialDropout2D ' 'or Dropout, passed as a string.')
    
    input1   = Input(shape = (Chans, Samples, 1))

    ##################################################################
    block1       = Conv2D(F1, (1, kernLength), padding = 'same', input_shape = (Chans, Samples, 1), use_bias = False)(input1)
    block1       = BatchNormalization()(block1)

    block1       = DepthwiseConv2D((Chans, 1), use_bias = False, depth_multiplier = D, depthwise_constraint = max_norm(1.))(block1)
    block1       = BatchNormalization()(block1)
    block1       = Activation('elu')(block1)
    block1       = AveragePooling2D((1, 4))(block1)
    block1       = dropoutType(dropoutRate)(block1)


    block2       = SeparableConv2D(F2, (1, 16), use_bias = False, padding = 'same')(block1)
    block2       = BatchNormalization()(block2)
    block2       = Activation('elu')(block2)
    block2       = AveragePooling2D((1, 8))(block2)
    block2       = dropoutType(dropoutRate)(block2)

    flatten      = Flatten(name = 'flatten')(block2)
    
    dense        = Dense(nb_classes, name = 'dense', kernel_constraint = max_norm(norm_rate))(flatten)
    softmax      = Activation('softmax', name = 'softmax')(dense)
    
    return Model(inputs=input1, outputs=softmax)





In [8]:
model=EEGNet(4, 22, 1001, kernLength=125)


In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)

y_val = le.transform(y_val)
y_test = le.transform(y_test)

from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=4)
y_val = to_categorical(y_val, num_classes=4)
y_test = to_categorical(y_test, num_classes=4)

print(model.output_shape)
print(y_train.shape)


(None, 4)
(3456, 4)


In [11]:

# convert data to NHWC (trials, channels, samples, kernels) format. Data 
# contains 60 channels and 151 time-points. Set the number of kernels to 1.
X_train      = X_train.reshape(X_train.shape[0], chans, samples, kernels)
X_val   = X_val.reshape(X_val.shape[0], chans, samples, kernels)
X_test       = X_test.reshape(X_test.shape[0], chans, samples, kernels)
   
print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

# configure the EEGNet-8,2,16 model with kernel length of 32 samples (other 
# model configurations may do better, but this is a good starting point)
model = EEGNet(nb_classes = 4, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 32, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

# compile the model and set the optimizers
model.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])

# count number of parameters in the model
numParams    = model.count_params()    

# set a valid path for your system to record model checkpoints
checkpointer = ModelCheckpoint(filepath='/tmp/checkpoint.h5', verbose=1,
                               save_best_only=True)



# the syntax is {class_1:weight_1, class_2:weight_2,...}. Here just setting
# the weights all to be 1
class_weights = {0:1, 1:1, 2:1, 3:1}

################################################################################
# fit the model. Due to very small sample sizes this can get
# pretty noisy run-to-run, but most runs should be comparable to xDAWN + 
# Riemannian geometry classification (below)
################################################################################
fittedModel = model.fit(X_train, y_train, batch_size = 16, epochs = 100, 
                        verbose = 2, validation_data=(X_val, y_val),
                        callbacks=[checkpointer], class_weight = class_weights)



###############################################################################
# make prediction on test set.
###############################################################################

probs       = model.predict(X_test)
preds       = probs.argmax(axis = -1)  
acc         = np.mean(preds == y_test.argmax(axis=-1))
print("Classification accuracy: %f " % (acc))

X_train shape: (3456, 22, 1001, 1)
3456 train samples
576 test samples
Epoch 1/100

Epoch 00001: val_loss improved from inf to 1.34537, saving model to /tmp/checkpoint.h5
216/216 - 12s - loss: 1.3649 - accuracy: 0.3142 - val_loss: 1.3454 - val_accuracy: 0.3030
Epoch 2/100

Epoch 00002: val_loss improved from 1.34537 to 1.25118, saving model to /tmp/checkpoint.h5
216/216 - 12s - loss: 1.3186 - accuracy: 0.3634 - val_loss: 1.2512 - val_accuracy: 0.3958
Epoch 3/100

Epoch 00003: val_loss improved from 1.25118 to 1.22567, saving model to /tmp/checkpoint.h5
216/216 - 12s - loss: 1.2814 - accuracy: 0.3918 - val_loss: 1.2257 - val_accuracy: 0.4219
Epoch 4/100

Epoch 00004: val_loss improved from 1.22567 to 1.17758, saving model to /tmp/checkpoint.h5
216/216 - 12s - loss: 1.2645 - accuracy: 0.4091 - val_loss: 1.1776 - val_accuracy: 0.4462
Epoch 5/100

Epoch 00005: val_loss did not improve from 1.17758
216/216 - 12s - loss: 1.2516 - accuracy: 0.4181 - val_loss: 1.2055 - val_accuracy: 0.3915
Epo

KeyboardInterrupt: 